In [7]:
import pandas as pd
import geopandas as gpd
import getpass
from pathlib import Path

user = getpass.getuser()
PROJECT_ROOT = Path(f"/Users/{user}/Final_Lawyer_Git July10")
# Load data
# FIPS
fips_path = PROJECT_ROOT/"Data/Geography/FIPS/US states FIPS.csv"   # FIPS path
df_fips = pd.read_csv(fips_path)  # Load FIPS
df_fips["FIPS Code"] = df_fips["FIPS Code"].astype(str).str.zfill(2)    # Ensure string, add 0 to single digits

# MSAs
msa_shapefile_path = PROJECT_ROOT/"Data/Geography/CBSA_shapefile_2025/tl_2025_us_cbsa.shp"    # Shapefile path
gdf_msa = gpd.read_file(msa_shapefile_path) # Load MSA shapefile

gdf_msa = gdf_msa[gdf_msa["LSAD"] == 'M1'].reset_index(drop=True)   # Remove micro
gdf_msa['State_Abbr'] = gdf_msa['NAME'].str[-2:]    # Get state abbriviations
gdf_msa = gdf_msa[gdf_msa['State_Abbr'].isin(df_fips['Postal Abbr.'])].copy()   # Filter only US states
valid_geoid = gdf_msa["GEOID"].astype(str).tolist()  # GEOID list

# Lawyers
lawyers_path = PROJECT_ROOT/"Data/BrightData_Lawyers/BrightData_Lawyers_master_normalized_1overN.csv" # Lawyers data Path
lawyers_data = pd.read_csv(lawyers_path)   # Load data
lawyers_data.rename(columns={'CBSA': 'AREA'}, inplace=True) # Rename columns
lawyers_data["AREA"] = lawyers_data["AREA"].astype("Int64").astype(str)
lawyers_data = lawyers_data[lawyers_data["AREA"].isin(valid_geoid)]   # Take only MSAs from the list

# Proxies
# Immigration lawyers
foreign_born_path = PROJECT_ROOT/"Data/Proxies/Immigration/ACSDP1Y2024.DP02-Data.csv" # Foreign born data Path
foreign_born_data = pd.read_csv(foreign_born_path)   # Load data

foreign_born_data = foreign_born_data[['GEO_ID', 'NAME', 'DP02_0094E']].copy()  # Take relevant columns
foreign_born_data.rename(columns={'GEO_ID': 'AREA', 'NAME': 'MSA', 'DP02_0094E': 'Foreign_Born_Population'}, inplace=True) # Rename columns

foreign_born_data = foreign_born_data.iloc[1:].reset_index(drop=True) # Remove first row

foreign_born_data['AREA'] = foreign_born_data['AREA'].str[-5:]    # Get GEOID
foreign_born_data["AREA"] = foreign_born_data["AREA"].astype(str) # GEOID as string

foreign_born_data = foreign_born_data[foreign_born_data["AREA"].isin(valid_geoid)]   # Take only MSAs from the list

foreign_born_data = foreign_born_data.merge(lawyers_data[['AREA', 'Immigration Law_normalized_1overN_count']], on=['AREA'], how='left')
foreign_born_data = foreign_born_data.dropna(subset=['Immigration Law_normalized_1overN_count'])
foreign_born_data = foreign_born_data.rename(columns={"Immigration Law_normalized_1overN_count": "Immigration"})
foreign_born_data = foreign_born_data.drop(columns=["MSA", "MSA_Name", "msa_name"], errors="ignore")

foreign_born_data.to_csv( PROJECT_ROOT/"Data/Proxies/Immigration/Immigration_Proxy_Normalized.csv", index=False)